[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedelshazly27/mizan-kuwait-energy-toolkit/blob/main/notebooks/01-quickstart.ipynb)

In [1]:
# On Colab: clone the repository and install it (inputs are committed, no credentials needed).
import os, sys
if "google.colab" in sys.modules and not os.path.exists("mizan-kuwait-energy-toolkit"):
    !git clone --depth 1 https://github.com/ahmedelshazly27/mizan-kuwait-energy-toolkit.git
    %pip install -q -e "mizan-kuwait-energy-toolkit[pv,econ,system]" matplotlib
    os.chdir("mizan-kuwait-energy-toolkit")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from kwtk import viz; viz.style()

# 1. Quickstart: evaluate a plant

A plant is a YAML file. `validate` checks it and explains problems in sentences;
`evaluate` simulates it hour by hour under Kuwaiti weather, costs it, and places it
in Kuwait's power system. Every input is listed with its origin.

In [2]:
from kwtk.config.validate import validate
print(validate('configs/examples/shagaya-scale-pv-tracker.yaml').text())

valid (4 defaults applied; see the run report)


A broken file fails with sentences, not a stack trace:

In [3]:
import yaml, tempfile
cfg = yaml.safe_load(open('configs/examples/shagaya-scale-pv-tracker.yaml'))
cfg['inverters']['dc_ac_ratio'] = 2.5
cfg['site']['location']['lat'] = 31.0
cfg['pv']['arrays'][0]['layout']['gcr'] = 0.33
cfg['pv']['arrays'][0]['mountng'] = cfg['pv']['arrays'][0].pop('mounting')
f = tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False); yaml.safe_dump(cfg, f); f.close()
print(validate(f.name).text())

error: site.location.lat is 31.0; it must be at most 30.15
error: pv.arrays.0.mounting is required
error: pv.arrays.0.mountng is not a field of this schema (check the spelling)


In [4]:
from kwtk.plant.evaluate import evaluate
run = evaluate('configs/examples/shagaya-scale-pv-tracker.yaml')
r = run['result']
pd.DataFrame(r['energy_by_year']).set_index('year').round(1)

,pv_ac_mwh,export_mwh,curtailed_mwh,battery_discharge_mwh,battery_charge_mwh,aux_mwh,import_mwh,inverter_clipping_mwh,capacity_factor_ac,specific_yield_kwh_per_kwp
year,,,,,,,,,,
2019,3097397.6,3097397.6,0.0,0.0,0.0,0.0,0.0,4527.3,0.3,2167.6
2020,3152722.0,3152722.0,0.0,0.0,0.0,0.0,0.0,2173.9,0.3,2206.3
2021,3165839.7,3165839.7,0.0,0.0,0.0,0.0,0.0,2311.2,0.3,2215.5
2022,3145605.9,3145605.9,0.0,0.0,0.0,0.0,0.0,4704.3,0.3,2201.3
2023,3098943.3,3098943.3,0.0,0.0,0.0,0.0,0.0,6065.2,0.3,2168.6


In [5]:
e = r['economics']
print(f"LCOE (real): {e['lcoe_real_fils_per_kwh']:.1f} fils/kWh  ({e['lcoe_real_usd_per_kwh']*100:.2f} US cents)")
print(f"System: fuel cost avoided {r['system_context']['fuel_cost_avoided_kd_per_mwh']:.1f} KD/MWh, "
      f"curtailed by the system {r['system_context']['system_curtailment_mwh']/1e3:.0f} GWh/yr")
print('Unverified inputs:', ', '.join(x.split(':')[1] for x in r['unverified_inputs']))

LCOE (real): 17.2 fils/kWh  (5.62 US cents)
System: fuel cost avoided 25.7 KD/MWh, curtailed by the system 38 GWh/yr
Unverified inputs: albedo.desert, availability.plant, cleaning.cost, degradation.pv_annual, degradation.pv_first_year, fx.kwd_per_usd, losses.dc_mismatch_pct, losses.dc_wiring_pct, soiling.kuwait_rate_by_month, wacc.kuwait_ipp


Every input and where it came from:

In [6]:
pd.DataFrame(r['inputs']).T[['value', 'origin', 'status']].fillna('')

,value,origin,status
site.soiling_source,"[0.49, 0.49, 0.49, 0.49, 0.49, 0.49, 0.49, 0.4...",assumption:soiling.kuwait_rate_by_month,unverified
soiling.rain_cleaning_threshold,6.0,assumption:soiling.rain_cleaning_threshold,proxy
soiling.max_loss,30.0,assumption:soiling.max_loss,proxy
om.cleaning.effectiveness,1.0,assumption:cleaning.effectiveness,proxy
pv.arrays[a1].module,jinko-jkm315pp-72,library:jinko-jkm315pp-72,
pv.arrays[a1].layout.collector_width_m,1.924,configuration,
site.terrain.albedo,0.3,assumption:albedo.desert,unverified
pv.arrays[a1].thermal_model.u_c,29.0,assumption:thermal.pvsyst_uc_tracker,proxy
pv.arrays[a1].thermal_model.u_v,0.0,assumption:thermal.pvsyst_uv,proxy
pv.arrays[a1].losses.dc_mismatch_pct,2.0,assumption:losses.dc_mismatch_pct,unverified
